# Notebook 5: Community Detection and Aggregation

This notebook processes the attribute-enriched graph **G2** to detect structural communities using the **Leiden algorithm**.

For each community, the LLM is invoked to generate a **community summary** using all semantic units within that community.  
If a community contains only one semantic unit, summarization is skipped.  
These summaries form the content for **High-level Element (H) nodes**.

From each community summary, an additional LLM call extracts **keywords** and **titles**.  
These results serve as the content for **High-level Overview (O) nodes**.

The resulting communities are also saved for future processing.

In [1]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

The directory of this script is: c:\Users\HP\Desktop\Projects\NodeRAG\graphs
The root directory is: c:\Users\HP\Desktop\Projects\NodeRAG


In [2]:
import sys
sys.path.append(root_path)
from graphs.Node import Node

In [3]:
import pickle
with open(f"{root_path}/graphs/data/graphs/G2_medical_attribute_enriched_graph.pkl", "rb") as f:
    medical_g2 = pickle.load(f)

In [4]:
import igraph as ig
from leidenalg import find_partition, ModularityVertexPartition
from collections import defaultdict

def leiden_community_detection(node_dict):
    id_to_idx = {nid: i for i, nid in enumerate(node_dict.keys())}
    idx_to_id = {v: k for k, v in id_to_idx.items()}

    edge_weights = defaultdict(float)
    for nid, node in node_dict.items():
        for neighbor, weight in node.edges.items():
            a, b = id_to_idx[nid], id_to_idx[neighbor]
            if a < b:
                edge_weights[(a, b)] += weight

    edges = list(edge_weights.keys())
    weights = list(edge_weights.values())

    g = ig.Graph(n=len(node_dict), edges=edges, directed=False)
    g.es["weight"] = weights

    partition = find_partition(g, ModularityVertexPartition, weights="weight")
    membership = partition.membership
    communities = {idx_to_id[i]: membership[i] for i in range(len(node_dict))}
    return communities


In [5]:
from collections import Counter
medical_communities = leiden_community_detection(medical_g2)
medical_communities_counts = Counter(medical_communities.values())
s1 = 0
for val in medical_communities_counts.values():
    if val==1:
        s1+=val
print("Medical single-node communities:", s1)

Medical single-node communities: 588


In [6]:
medical_communities_counts

Counter({0: 2022,
         1: 1778,
         2: 1588,
         3: 1578,
         4: 1527,
         5: 1448,
         6: 1427,
         7: 1317,
         8: 1294,
         9: 1236,
         10: 1200,
         11: 1195,
         12: 1182,
         13: 1105,
         14: 1097,
         15: 1074,
         16: 979,
         17: 963,
         18: 961,
         19: 852,
         20: 847,
         21: 797,
         22: 705,
         23: 648,
         24: 591,
         25: 374,
         26: 354,
         27: 260,
         28: 244,
         29: 232,
         30: 216,
         31: 211,
         32: 90,
         33: 51,
         34: 25,
         35: 16,
         36: 11,
         38: 10,
         37: 10,
         39: 9,
         40: 8,
         41: 4,
         42: 3,
         43: 1,
         44: 1,
         45: 1,
         46: 1,
         47: 1,
         48: 1,
         49: 1,
         50: 1,
         51: 1,
         52: 1,
         53: 1,
         54: 1,
         55: 1,
         56: 1,
         57

In [7]:
communities_medical = defaultdict(list)
for node_id, community_id in medical_communities.items():
    #if medical_g2[node_id].node_type == "N":
    #    continue
    communities_medical[community_id].append(node_id)

In [8]:
communities_medical

defaultdict(list,
            {22: ['medical-0-S-0',
              'medical-0-S-0-R-1',
              'medical-15-S-3-R-10',
              'medical-15-S-5',
              'medical-15-S-5-R-0',
              'medical-15-S-5-R-1',
              'medical-15-S-5-R-2',
              'medical-16-S-1',
              'medical-16-S-1-R-0',
              'medical-16-S-1-R-1',
              'medical-16-S-1-R-2',
              'medical-16-S-1-R-3',
              'medical-16-S-1-R-4',
              'medical-16-S-1-R-5',
              'medical-16-S-1-R-7',
              'medical-16-S-1-R-9',
              'medical-16-S-1-R-11',
              'medical-20-S-4-R-5',
              'medical-32-S-0-R-8',
              'medical-33-S-1-R-3',
              'medical-35-S-1',
              'medical-35-S-1-R-0',
              'medical-35-S-1-R-2',
              'medical-35-S-1-R-3',
              'medical-64-S-0-R-0',
              'medical-118-S-2-R-13',
              'medical-118-S-2-R-14',
              'med

In [ ]:
from google import genai

with open(f"{root_path}/API_KEY.txt", "r", encoding="utf-8") as f:
    API_KEY = f.read()
    
def call_gemini(text):
    client = genai.Client(api_key = API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=text,
    )
    return response.text

In [10]:
import time
from tqdm import tqdm

In [11]:
medical_communities_response = {}

In [16]:
from graphs.prompt.high_level_elements_generation_prompt import high_level_elements_generation_prompt

for c in tqdm(communities_medical):
    if f'medical-H-{c}' in medical_communities_response:
        continue
    semantic_count = 0
    nodes_id = communities_medical[c]
    content = []
    for nid in nodes_id:
        node = medical_g2[nid]
        if node.node_type == "S":
            semantic_count+=1
        if node.node_type in ["R", "N"]:
            continue
        content.append(node.content)
    if semantic_count < 2:
        continue
    else:
        content = "\n".join(content)
        response = call_gemini(high_level_elements_generation_prompt(content))
        medical_communities_response[f'medical-H-{c}'] = response
        time.sleep(2)

100%|██████████| 631/631 [01:37<00:00,  6.45it/s]


In [17]:
medical_communities_overview = {}

In [18]:
from graphs.prompt.high_level_overview_generation_prompt import high_level_overview_prompt
for element_id, element in tqdm(medical_communities_response.items()):
    overview_id = f"{element_id}-O-0"
    if overview_id in medical_communities_overview:
        continue
    overview = call_gemini(high_level_overview_prompt(element))
    medical_communities_overview[overview_id] = overview
    time.sleep(4)

100%|██████████| 37/37 [03:02<00:00,  4.94s/it]


In [19]:
import pandas as pd
medical_df = pd.DataFrame(list(medical_communities_response.items()), columns=["node_id", "community_summary"])
medical_df.to_parquet("data//nodes/community/medical_communities_summary.parquet")
medical_df = pd.read_parquet("data/nodes/community/medical_communities_summary.parquet")
medical_df

,node_id,community_summary
0,medical-H-22,Here are the distinct categories of high-level...
1,medical-H-24,Here are distinct categories of high-level inf...
2,medical-H-21,Here are distinct categories of high-level inf...
3,medical-H-15,Here are the distinct categories of high-level...
4,medical-H-2,Here's a breakdown of distinct high-level info...
5,medical-H-14,Here are the distinct categories of high-level...
6,medical-H-10,Here are distinct categories of high-level inf...
7,medical-H-5,Here are the distinct categories of high-level...
8,medical-H-11,Here's a breakdown of the distinct categories ...
9,medical-H-8,Here's a breakdown of distinct categories of h...


In [20]:
medical_overview = pd.DataFrame(list(medical_communities_overview.items()), columns=["node_id", "community_overview"])
medical_overview.to_parquet("data/nodes/community/medical_communities_overview.parquet")
medical_overview = pd.read_parquet("data/nodes/community/medical_communities_overview.parquet")
medical_overview

,node_id,community_overview
0,medical-H-22-O-0,HPV Cancer Etiology Diagnosis Screening Vaccin...
1,medical-H-24-O-0,"Skin Cancer: Origins, Risk Factors, Appearance..."
2,medical-H-21-O-0,"Cancer Patient Empowerment, Guidance, Genetics..."
3,medical-H-15-O-0,Cancer genetics molecular diagnostics biomarke...
4,medical-H-2-O-0,"Cancer Diagnosis, Staging, Treatment, and Prog..."
5,medical-H-14-O-0,"Cancer Treatment, Fertility, Side Effects, Dia..."
6,medical-H-10-O-0,Cancer Diagnosis Treatment Planning Testing Me...
7,medical-H-5-O-0,Cancer Patient Journey Management
8,medical-H-11-O-0,"Cancer Diagnosis, Staging, Grading, Biomarkers..."
9,medical-H-8-O-0,"Endocrine System, Hormone Function, Adrenal Gl..."


In [21]:
with open(f"{root_path}/graphs/data/nodes/community/medical_communities.pkl", "wb") as f:
    pickle.dump(communities_medical, f)
with open(f"{root_path}/graphs/data/nodes/community/medical_communities.pkl", "rb") as f:
    communities_medical_loaded = pickle.load(f)

In [22]:
communities_medical_loaded

defaultdict(list,
            {22: ['medical-0-S-0',
              'medical-0-S-0-R-1',
              'medical-15-S-3-R-10',
              'medical-15-S-5',
              'medical-15-S-5-R-0',
              'medical-15-S-5-R-1',
              'medical-15-S-5-R-2',
              'medical-16-S-1',
              'medical-16-S-1-R-0',
              'medical-16-S-1-R-1',
              'medical-16-S-1-R-2',
              'medical-16-S-1-R-3',
              'medical-16-S-1-R-4',
              'medical-16-S-1-R-5',
              'medical-16-S-1-R-7',
              'medical-16-S-1-R-9',
              'medical-16-S-1-R-11',
              'medical-20-S-4-R-5',
              'medical-32-S-0-R-8',
              'medical-33-S-1-R-3',
              'medical-35-S-1',
              'medical-35-S-1-R-0',
              'medical-35-S-1-R-2',
              'medical-35-S-1-R-3',
              'medical-64-S-0-R-0',
              'medical-118-S-2-R-13',
              'medical-118-S-2-R-14',
              'med